In [1]:
# import the functions from collector.py
import sys
import os
sys.path.append(os.path.abspath(".."))

from bjarne_api.collector_new import Fetcher
import pandas as pd

In [2]:
### FILTER FUNCTION FOR ICE/IC TRAINS ONLY ###
def filter_train_type(df):
    output_df = df[df["train_type"].str.contains("ICE|IC", case=False, na=False)].copy()
    return output_df

In [3]:
### FUNCTION TO FIND POSSIBLE DESTINATIONS FROM A GIVEN STATION ###
def get_possible_destinations(df, station_name):
    possible_destinations = []

    for ride_id, train_group in df.groupby("ride_id"):
        
        # find row where current_station is station_name
        start_row = train_group[train_group["station_current"] == station_name]
        
        # If the train actually stops at our station:
        if not start_row.empty:
            station_time = start_row["departure_real"].iloc[0]
            
            # get stations where actual_arrival is greater than station_time
            later_stops = train_group[train_group["arrival_real"] > station_time]
            
            # get station names and add to possible_destinations
            stations_names = later_stops["station_current"].tolist()
            possible_destinations.extend(stations_names)
            
    # use set() to remove duplicates
    return sorted(list(set(possible_destinations)))

In [4]:
import pandas as pd

### FUNCTION TO GET CONNECTION BETWEEN TWO STATIONS ###
def get_connections(df, station_start, station_dest):
    
    suited_rides = []

    for ride_id, train_group in df.groupby("ride_id"):
        start_row = train_group[train_group["station_current"] == station_start]
        
        if not start_row.empty:
            station_start_time = start_row["departure_real"].iloc[0]
            
            # get stations coming after station_start_time
            later_stops = train_group[train_group["arrival_real"] > station_start_time]
            
            # check: is station_dest in later_stops?
            if station_dest in later_stops["station_current"].values:
                # add ride_id to suited_rides
                suited_rides.append(train_group)

    # combine all suited_rides into one df
    if suited_rides:
        df_trip = pd.concat(suited_rides, ignore_index=True)
        return df_trip
    else:
        print(f"No current connections found.")
        return None

In [ ]:
### DEFINE JOURNEY
input = "Göttingen" 

### GET INFOS ON ALL RIDES
# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = input
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")


# filter data
df_filtered = filter_train_type(final_df)

# get valid destinations (list format)
ls_destinations = get_possible_destinations(df_filtered, input)
# select destination
destination = "Berlin Hbf" # COMES FROM INPUT IN STREAMLIT

# get rides between stations
df_trip = get_connections(df_filtered, input, destination)
df_trip = df_trip.drop(columns = ["current_delay", "stops_total", "stop_index", "stops_remaining"])

Suche Station ID für Göttingen...
Station ID gefunden: 8000128. Suche Fernverkehrsabfahrten (nächste 60min)...
9 Abfahrten gefunden. Lade Details...
Lade Trip 1/9: ICE 995
Lade Trip 2/9: ICE 883
Lade Trip 3/9: ICE 794
Lade Trip 4/9: ECE 9
Lade Trip 5/9: ICE 576
Lade Trip 6/9: ICE 379
Lade Trip 7/9: ICE 586
Lade Trip 8/9: ICE 683
Lade Trip 9/9: ICE 2849

--- FERTIG ---
     ride_id train_name train_type      station_start   station_dest  \
0          1    ICE 995        ICE  Berlin Ostbahnhof  Stuttgart Hbf   
1          1    ICE 995        ICE  Berlin Ostbahnhof  Stuttgart Hbf   
2          1    ICE 995        ICE  Berlin Ostbahnhof  Stuttgart Hbf   
3          1    ICE 995        ICE  Berlin Ostbahnhof  Stuttgart Hbf   
4          1    ICE 995        ICE  Berlin Ostbahnhof  Stuttgart Hbf   
..       ...        ...        ...                ...            ...   
105        9   ICE 2849        ICE    Hamburg Dammtor    München Hbf   
106        9   ICE 2849        ICE    Hamburg Dammtor